## 01.joblib

모델 성능 개선 수업에서는 교차검증과 하이퍼파라미터 튜닝으로 더 좋은 모델을 고름.
하지만 좋은 모델을 찾은 뒤에도 매번 다시 학습하면 시간이 오래 걸리고, 서비스나 프로젝트에서 재사용하기 어려움.

**배우는 이유**
- 학습이 끝난 모델을 파일로 저장해 재사용하기 위함.
- 프로젝트 발표나 배포에서 같은 모델을 다시 불러와 예측하기 위함.
- 전처리 기준과 모델을 함께 저장해 예측 실수를 줄이기 위함.

**어디서 사용하는가?**
- Streamlit/Django/FastAPI 같은 앱에서 학습된 모델을 불러와 예측할 때.
- 모델 학습 시간이 길어 매번 다시 학습하기 어려울 때.
- GridSearchCV로 찾은 최적 모델을 보관할 때.
- 팀 프로젝트에서 학습 담당자와 서비스 담당자가 모델 파일을 공유할 때.

**핵심 용어**
- `dump`: Python 객체를 파일로 저장함.
- `load`: 파일에 저장된 Python 객체를 다시 불러옴.
- `Pipeline`: 전처리와 모델을 하나의 학습/예측 흐름으로 묶음.
- `best_estimator_`: GridSearchCV가 교차검증 기준으로 선택한 최종 모델 객체임.

## 02. 실습 환경 준비

이번 실습에서는 Wine 다중분류 데이터를 사용한다.
`StandardScaler`, `LogisticRegression`, `GridSearchCV`, `Pipeline`, `joblib`을 함께 사용해 모델 개선 후 저장까지 연결한다.

`joblib`은 별도 Python 라이브러리이므로 원칙적으로 설치가 필요하다.
다만 머신러닝 수업 환경에서는 보통 `scikit-learn`을 설치할 때 의존성으로 함께 설치되어 있어, 이미 `scikit-learn`이 정상 동작하면 대부분 바로 사용할 수 있다.

```python
import joblib
```

위 import에서 오류가 나면 아래 명령으로 설치한다.

```python
%pip install joblib
```

터미널에서는 다음 명령을 사용할 수 있다.

```bash
pip install joblib
```

In [1]:
# %pip install joblib

In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

## 03. Wine 데이터 로드와 학습/평가 분리

Wine 데이터는 와인의 화학 성분을 바탕으로 3개 품종 중 하나를 예측하는 분류 데이터셋이다.
수치 feature의 범위가 서로 다르므로, 뒤에서 `StandardScaler`를 사용해 스케일을 맞춤.

In [3]:
wine = load_wine(as_frame=True)

wine_X = wine.data
wine_y = wine.target

wine_X_train, wine_X_test, wine_y_train, wine_y_test = train_test_split(
    wine_X,
    wine_y,
    test_size=0.2,
    random_state=42,
    stratify=wine_y
)

print('train:', wine_X_train.shape, wine_y_train.shape)
print('test:', wine_X_test.shape, wine_y_test.shape)
print('target names:', wine.target_names)
display(wine_X_train.head())


train: (142, 13) (142,)
test: (36, 13) (36,)
target names: ['class_0' 'class_1' 'class_2']


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
36,13.28,1.64,2.84,15.5,110.0,2.60,2.68,0.34,1.36,4.60,1.09,2.78,880.0
30,13.73,1.50,2.70,22.5,101.0,3.00,3.25,0.29,2.38,5.70,1.19,2.71,1285.0
26,13.39,1.77,2.62,16.1,93.0,2.85,2.94,0.34,1.45,4.80,0.92,3.22,1195.0
12,13.75,1.73,2.41,16.0,89.0,2.60,2.76,0.29,1.81,5.60,1.15,2.90,1320.0
148,13.32,3.24,2.38,21.5,92.0,1.93,0.76,0.45,1.25,8.42,0.55,1.62,650.0


## 04. Pipeline으로 전처리와 모델 묶기

모델을 저장할 때 모델 객체만 저장하면, 예측 전에 같은 스케일링을 적용해야 한다는 사실을 놓치기 쉬움.
`Pipeline`을 사용하면 `StandardScaler`와 `LogisticRegression`을 하나로 묶어 저장할 수 있다.

In [4]:
baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000))
])

# 학습
baseline_pipeline.fit(wine_X_train, wine_y_train)

# 예측
baseline_pred = baseline_pipeline.predict(wine_X_test)

# 평가
print('baseline accuracy:', accuracy_score(wine_y_test, baseline_pred))
print('baseline f1-macro:', f1_score(wine_y_test, baseline_pred, average='macro'))

baseline accuracy: 0.9722222222222222
baseline f1-macro: 0.9709618874773139


## 05. GridSearchCV로 최종 모델 후보 선택

모델 성능 개선 단원과 연결해 `GridSearchCV`로 Logistic Regression의 `C` 값을 비교한다.
`C`는 규제 강도의 역수라서 값이 작으면 규제가 강하고, 값이 크면 규제가 약한다.

In [5]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    refit=True
)

grid_search.fit(wine_X_train, wine_y_train)

best_pipeline = grid_search.best_estimator_
best_pred = best_pipeline.predict(wine_X_test)

result_df = pd.DataFrame(grid_search.cv_results_)
result_df = result_df[[
    'param_model__C',
    'param_model__class_weight',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]].sort_values('rank_test_score')

display(result_df.round(4))
print('best_params:', grid_search.best_params_)
print('best_cv_accuracy:', round(grid_search.best_score_, 4))
print('test_accuracy:', round(accuracy_score(wine_y_test, best_pred), 4))
print('test_macro_f1:', round(f1_score(wine_y_test, best_pred, average='macro'), 4))

,param_model__C,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
2,0.10,NaN,0.9931,0.0138,1
4,1.00,NaN,0.9931,0.0138,1
3,0.10,balanced,0.9862,0.0276,3
5,1.00,balanced,0.9860,0.0172,4
6,10.00,NaN,0.9860,0.0172,4
7,10.00,balanced,0.9860,0.0172,4
0,0.01,NaN,0.9791,0.0277,7
1,0.01,balanced,0.9719,0.0259,8


best_params: {'model__C': 0.1, 'model__class_weight': None}
best_cv_accuracy: 0.9931
test_accuracy: 1.0
test_macro_f1: 1.0


## 06. joblib.dump로 최종 Pipeline 저장

`joblib.dump()`는 Python 객체를 파일로 저장한다.
여기서는 전처리 기준과 모델이 함께 들어 있는 `best_pipeline`을 저장한다.

In [6]:
current_dir = Path.cwd()
print(current_dir)

# model을 저장할 경로
model_dir = current_dir / 'models'
print(model_dir)

# models 폴더 생성(이미 존재해도 오류 X)
model_dir.mkdir(exist_ok=True)

# 저장할 모델 파일명, 메타데이터 저장 파일명 지정
model_path = model_dir / 'wine_logistic_pipeline.joblib'
metadata_path = model_dir / 'wine_logistic_pipeline_metadata.joblib'

# joblib.dump(): 파이썬 객체(모델)을 지정한 파일 경로에 저장
save_files = joblib.dump(best_pipeline, model_path)

# 메타데이터 구성
# - 모델에 대한 설명서 역할
# - 재학습의 기준 역할
model_metadata = {
    'feature_names': wine_X.columns.to_list(),
    'target_names': wine.target_names.tolist(),
    'best_params': grid_search.best_params_,
    'cv_score': float(grid_search.best_score_)
}
joblib.dump(model_metadata, metadata_path)


C:\SKN_AI\06_machine_learning\08_model_performance
C:\SKN_AI\06_machine_learning\08_model_performance\models


['C:\\SKN_AI\\06_machine_learning\\08_model_performance\\models\\wine_logistic_pipeline_metadata.joblib']

## 07. joblib.load로 불러와 예측 확인

`joblib.load()`는 저장된 객체를 다시 메모리로 불러온다.
불러온 모델이 저장 전 모델과 같은 예측을 내는지 확인하면 저장/로드가 정상인지 판단할 수 있다.

In [7]:
loaded_pipeline = joblib.load(model_path) # 학습이 완료된 모델
loaded_metadata = joblib.load(metadata_path)

# 불러온 모델로 test 데이터 결과 예측
loaded_pred = loaded_pipeline.predict(wine_X_test)

# 평가
print('loaded accuracy:', accuracy_score(wine_y_test, loaded_pred))

# 메타데이터 확인
print('loaded_metadata:', loaded_metadata)

loaded accuracy: 1.0
loaded_metadata: {'feature_names': ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline'], 'target_names': ['class_0', 'class_1', 'class_2'], 'best_params': {'model__C': 0.1, 'model__class_weight': None}, 'cv_score': 0.993103448275862}


## 08. 새 데이터 예측 흐름

저장된 모델은 새 데이터가 들어왔을 때 바로 예측에 사용된다.
이때 새 데이터도 학습 때 사용한 feature 이름과 순서를 맞춰야 한다.

In [8]:
new_samples = wine_X_test.iloc[:5].copy()

new_pred = loaded_pipeline.predict(new_samples)
new_proba = loaded_pipeline.predict_proba(new_samples)

prediction_df = new_samples.copy()
prediction_df['predicted_class'] = new_pred
prediction_df['predicted_name'] = [wine.target_names[i] for i in new_pred]
prediction_df['max_probability'] = new_proba.max(axis=1)

display(prediction_df)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,predicted_class,predicted_name,max_probability
10,14.10,2.16,2.30,18.0,105.0,2.95,3.32,0.22,2.38,5.75,1.25,3.17,1510.0,0,class_0,0.987474
134,12.51,1.24,2.25,17.5,85.0,2.00,0.58,0.60,1.25,5.45,0.75,1.51,650.0,2,class_2,0.535698
28,13.87,1.90,2.80,19.4,107.0,2.95,2.97,0.37,1.76,4.50,1.25,3.40,915.0,0,class_0,0.921620
121,11.56,2.05,3.23,28.5,119.0,3.18,5.08,0.47,1.87,6.00,0.93,3.69,465.0,1,class_1,0.655945
62,13.67,1.25,1.92,18.0,94.0,2.10,1.79,0.32,0.73,3.80,1.23,2.46,630.0,1,class_1,0.789720


## 09. Pipeline에 저장된 구성 확인

스케일링이 필요한 모델에서 모델만 따로 저장하면, 예측 전에 어떤 평균과 표준편차로 변환했는지 같이 관리해야 한다.
`Pipeline`은 이 문제를 줄여줌.

In [9]:
# 파이프라인에 저장된 각 단계를 이름으로 꺼내오기
loaded_scaler = loaded_pipeline.named_steps['scaler']
loaded_model = loaded_pipeline.named_steps['model']

print('Pipeline 단계: ', loaded_pipeline.named_steps.keys())
print('scaler가 기억한 feature의 개수: ', loaded_scaler.n_features_in_)
print('model이 학습한 class:', loaded_model.classes_)

Pipeline 단계:  dict_keys(['scaler', 'model'])
scaler가 기억한 feature의 개수:  13
model이 학습한 class: [0 1 2]


## 10. joblib 보안과 운영 주의사항

`joblib.load()`는 신뢰할 수 있는 파일에만 사용해야 한다.
joblib은 내부적으로 pickle 계열 직렬화를 사용하므로, 출처를 알 수 없는 파일을 불러오면 보안 위험이 생길 수 있다.

In [10]:
operation_notes = pd.DataFrame([
    {'항목': '모델 파일', '확인 내용': str(model_path)},
    {'항목': 'feature 개수', '확인 내용': len(loaded_metadata['feature_names'])},
    {'항목': 'target 이름', '확인 내용': ', '.join(loaded_metadata['target_names'])},
    {'항목': '최적 파라미터', '확인 내용': str(loaded_metadata['best_params'])},
    {'항목': '보안 주의', '확인 내용': '신뢰할 수 없는 joblib/pickle 파일은 load하지 않음'}
])

display(operation_notes)

,항목,확인 내용
0,모델 파일,C:\SKN_AI\06_machine_learning\08_model_perform...
1,feature 개수,13
2,target 이름,"class_0, class_1, class_2"
3,최적 파라미터,"{'model__C': 0.1, 'model__class_weight': None}"
4,보안 주의,신뢰할 수 없는 joblib/pickle 파일은 load하지 않음


## 11. 정리

- `joblib`은 학습된 scikit-learn 모델이나 Pipeline을 파일로 저장하고 다시 불러올 때 자주 사용함.
- 전처리가 필요한 모델은 모델만 저장하기보다 `Pipeline` 전체를 저장하는 편이 안전함.
- `GridSearchCV`로 튜닝했다면 `best_estimator_`를 최종 모델로 저장하면 됨.
- 저장 후에는 다시 불러와 같은 예측이 나오는지 확인해야 함.
- `joblib.load()`는 신뢰할 수 있는 파일에만 사용해야 함.